In [11]:

import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dormadum/gridlocktrain/train.csv
/kaggle/input/datasets/dormadum/gridlocktest/test.csv


In [12]:
df = pd.read_csv('/kaggle/input/datasets/dormadum/gridlocktrain/train.csv')

In [13]:
# Ensure the geohash column is treated as a string and handle missing values
df['geohash'] = df['geohash'].astype(str)

# 2. Check if ALL strings start with 'gp'
all_start_with_gp = df['geohash'].str.startswith('gp').all()
print(f"Do all geohashes start with 'gp'?: {all_start_with_gp}\n")

# 3. If they vary, count how many DO NOT start with 'gp'
# Create a boolean mask where True means it DOES NOT start with 'gp'
does_not_start_with_gp = ~df['geohash'].str.startswith('gp')
varying_count = does_not_start_with_gp.sum()

print(f"Number of geohashes that do NOT start with 'gp': {varying_count}")


Do all geohashes start with 'gp'?: False

Number of geohashes that do NOT start with 'gp': 77299


In [14]:
if varying_count > 0:
    print("\n--- Breakdown of variations ---")
    
    # Extract the first 2 characters of every geohash to see what prefixes exist
    df['prefix'] = df['geohash'].str[:2]
    print("1. Most common prefixes across the dataset:")
    print(df['prefix'].value_counts().head(10))
    
    print("\n2. Examples of geohashes that do not start with 'gp':")
    # Filter the dataframe to show the unique values that don't start with 'gp'
    non_gp_hashes = df[does_not_start_with_gp]['geohash'].value_counts()
    print(non_gp_hashes.head(10))
else:
    print("All geohashes strictly start with 'gp'!")


--- Breakdown of variations ---
1. Most common prefixes across the dataset:
prefix
qp    77299
Name: count, dtype: int64

2. Examples of geohashes that do not start with 'gp':
geohash
qp03wd    105
qp03wf    105
qp09t0    105
qp03w9    105
qp03x3    105
qp03wc    105
qp09ft    105
qp03x5    105
qp03we    105
qp03wg    105
Name: count, dtype: int64


In [15]:
print("--- Day Attribute Analysis ---")

# 1. Get all unique days sorted in order
unique_days = sorted(df['day'].unique())
print(f"Unique days present in dataset: {unique_days}")
print(f"Total number of unique days: {len(unique_days)}")

# 2. Check for missing gaps in the sequence
expected_sequence = list(range(min(unique_days), max(unique_days) + 1))
if unique_days == expected_sequence:
    print("✅ Perfect Sequence! No missing days in the range.")
else:
    missing_days = set(expected_sequence) - set(unique_days)
    print(f"⚠️ Gap Detected! Missing days: {sorted(list(missing_days))}")

# 3. Check if the dataset is ordered sequentially by day
# (This checks if it switches back and forth between days erratically)
day_changes = df['day'].diff().dropna()
# A change only happens when the difference between current row and previous row is not 0
num_times_day_switched = (day_changes != 0).sum()

print(f"\nNumber of times the day value changes across rows: {num_times_day_switched}")
print(f"Expected changes if perfectly grouped: {len(unique_days) - 1}")

if num_times_day_switched == len(unique_days) - 1:
    print("✅ The dataset is cleanly ordered block-by-block sequentially.")
else:
    print("ℹ️ The dataset rows are mixed/shuffled. (You might want to run `df = df.sort_values(by=['day', 'timestamp'])` later).")

# 4. Distribution of records per day (Check if data volume per day is regular/consistent)
print("\nNumber of records per day:")
print(df['day'].value_counts().sort_index())

--- Day Attribute Analysis ---
Unique days present in dataset: [np.int64(48), np.int64(49)]
Total number of unique days: 2
✅ Perfect Sequence! No missing days in the range.

Number of times the day value changes across rows: 1
Expected changes if perfectly grouped: 1
✅ The dataset is cleanly ordered block-by-block sequentially.

Number of records per day:
day
48    69427
49     7872
Name: count, dtype: int64


In [16]:
print(df['timestamp'].unique()[:15])
print(f"Total unique timestamps in a day: {df[df['day']==48]['timestamp'].nunique()}")

['0:0' '0:15' '0:30' '0:45' '1:0' '1:15' '1:30' '1:45' '2:0' '2:15' '2:30'
 '2:45' '3:0' '3:15' '3:30']
Total unique timestamps in a day: 96


In [18]:
train_hashes = set(df['geohash'].unique())
df_test = pd.read_csv('/kaggle/input/datasets/dormadum/gridlocktest/test.csv')
test_hashes = set(df_test['geohash'].unique())

print(f"Unique hashes in Train: {len(train_hashes)}")
print(f"Unique hashes in Test: {len(test_hashes)}")
print(f"New hashes in Test set: {len(test_hashes - train_hashes)}")

Unique hashes in Train: 1249
Unique hashes in Test: 1190
New hashes in Test set: 10


In [19]:
for col in ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks']:
    print(f"\n--- {col} Distribution (Train) ---")
    print(df[col].value_counts(dropna=False))
    print(f"--- {col} Distribution (Test) ---")
    print(df_test[col].value_counts(dropna=False))


--- RoadType Distribution (Train) ---
RoadType
Residential    69230
Street          3909
Highway         3560
NaN              600
Name: count, dtype: int64
--- RoadType Distribution (Test) ---
RoadType
Residential    33775
Highway         4272
Street          3407
NaN              324
Name: count, dtype: int64

--- Weather Distribution (Train) ---
Weather
Sunny    27717
Rainy    20824
Foggy    20243
Snowy     7718
NaN        797
Name: count, dtype: int64
--- Weather Distribution (Test) ---
Weather
Sunny    15078
Foggy    11102
Rainy    11081
Snowy     4086
NaN        431
Name: count, dtype: int64

--- LargeVehicles Distribution (Train) ---
LargeVehicles
Not Allowed    50673
Allowed        26626
Name: count, dtype: int64
--- LargeVehicles Distribution (Test) ---
LargeVehicles
Not Allowed    26178
Allowed        15600
Name: count, dtype: int64

--- Landmarks Distribution (Train) ---
Landmarks
Yes    52042
No     25257
Name: count, dtype: int64
--- Landmarks Distribution (Test) ---
Land

In [20]:
print(df['demand'].describe())
print(f"Skewness: {df['demand'].skew()}")

count    7.729900e+04
mean     9.394238e-02
std      1.421905e-01
min      6.245650e-07
25%      1.822723e-02
50%      4.775994e-02
75%      1.085951e-01
max      1.000000e+00
Name: demand, dtype: float64
Skewness: 3.7285172887316627
